In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [4]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [5]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [6]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [7]:
ground_truth[0]

{'question': 'I just found this course — is it too late to join now?',
 'document': '74eb249bbf'}

In [8]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [9]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='I just found this course — is it too late to join now?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"too late to join course late enrollment join now FAQ"}', call_id='call_5YcKFUmmN7K1Y0XAiAqnkCwI', name='search', type='function_call', id='fc_01b72332f35e1b50006a53ce142ccc8199acbc975b1f556b05', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_5YcKFUmmN7K1Y0XAiAqnkCwI',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still a

In [11]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [12]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"too late to join course late enrollment join now FAQ"}'}]

In [13]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [15]:
rec

{'question': 'I just found this course — is it too late to join now?',
 'document': '74eb249bbf'}

In [16]:
original_doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [17]:
answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [14]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'I just found this course — is it too late to join now?',
 'answer_agent': 'Yes — you can still join the course.\n\nAccording to the FAQ, if you just discovered it, you’re welcome to participate. The only caveat is that if you want a certificate, you need to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"too late to join course late enrollment join now FAQ"}'}],
 'cost': Decimal('0.00101625'),
 'document': '74eb249bbf'}

In [18]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [19]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [20]:
df_agent = pd.DataFrame(agent_answers)

In [22]:
df_agent.head()

,question,answer_agent,answer_orig,tool_calls,cost,document
0,I just found this course — is it too late to j...,Yes — you can still join.\n\nAccording to the ...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""to...",0.00109425,74eb249bbf
1,Can I still start the course if I'm coming in ...,Yes — you can start the course whenever you wa...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""la...",0.0012675,74eb249bbf
2,"If I enroll now, is there any chance to get a ...",Yes — you can still get a certificate **if you...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""en...",0.001341,74eb249bbf
3,What do I need to do to be eligible for the co...,"To be eligible for the course certificate, you...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""co...",0.0013425,74eb249bbf
4,Is the final project deadline the only thing t...,"No. For certification, the capstone project is...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""fi...",0.00133725,74eb249bbf


In [21]:
df_agent["cost"].sum()

Decimal('0.06412575')

In [23]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [ ]:
# !wget -O data/agent-answers.csv https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/data/agent-answers.csv

--2026-07-12 17:56:33--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/data/agent-answers.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 

200 OK
Length: 50628 (49K) [text/plain]
Saving to: ‘data/agent-answers.csv’

data/agent-answers. 100%[===================>]  49.44K  --.-KB/s    in 0.001s  

2026-07-12 17:56:33 (44.0 MB/s) - ‘data/agent-answers.csv’ saved [50628/50628]



In [26]:
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

In [27]:
df_agent.shape

(50, 6)